# H2 Worker thinking

Frozen before the run.

**Claim.** Latent steps help only when one worker holds both hops.

**Arms** (thinking off):
- K1 at m ∈ {0, 8, 16, 32}. m = 0 keeps the same worker wrapper, so the contrast is m alone.
- B1, as the reference.
- T1, as the positive control.
- All on toy hop1 and MuSiQue 2-hop, placed BB (P-colo, 0 crossings, both hops in the separately encoded second block) and AB (P-split, 1 crossing).

**Prediction.**
- K1(m) − K1(0) > 0 under BB.
- K1(m) − K1(0) is equivalent to 0 under AB (TOST ±2 EM).
- So the interaction gap(AB) − gap(BB) < 0 for the best m.

**Interpreting a null.**
- If T1 under BB ≈ B1, a worker that holds both hops can pre-compose in text. If K1 still doesn't gain under BB, the latent steps are what's broken, and that isn't the same claim as "thinking doesn't help".
- If T1 under BB is also far below B1, pre-composition isn't happening in any channel, and H2's premise fails at this model size.

**Pilot (Qwen3-4B HotpotQA, n = 50, router placement):** m = 10 .50 vs m = 0 .52. DeltaMAS (RWKV-7, GSM8K, n = 20): 0 steps 9/20 vs 30 steps 2/20. Qwen2.5-3B: latent steps produced word salad.

**Not built yet:**
- K3 (thought positions only).
- The worker self-answer health check: after m steps the worker answers from its own cache, to show whether the steps corrupt it.

In [ ]:
# Settings. Headless full run: see README "Running".
MODEL = "Qwen/Qwen3-4B"
DTYPE = "bfloat16"
SMOKE = True        # True: 2 questions per cell, to read raw outputs first. False: the full design.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from latent_dispatch import analysis, arms, data
from latent_dispatch.model import Model
from latent_dispatch.results import load_results, results_path, run
from latent_dispatch.score import contains_answer, normalize

## Questions and arms

Same questions at BB (colocated) and AB (split).

In [ ]:
qs = (data.load("toy", qtype="hop1", placement=["BB", "AB"])
      + data.load("musique", qtype="2hop", placement=["BB", "AB"]))
if SMOKE:
    qs = data.per_cell(qs, 2)
print(len(qs), "questions")

STEPS = [0, 8, 16, 32]
ARMS = {f"k1_m{m}": (arms.k1, {"m": m}) for m in STEPS}
ARMS["b1"] = (arms.b1, {})
ARMS["t1_1024"] = (arms.t1, {"report_tokens": 1024})     # positive control: can a worker pre-compose in text?

In [ ]:
model = Model.load(MODEL, dtype=DTYPE)
for name, (arm, options) in ARMS.items():
    run(model, qs, name, arm, **options)

In [ ]:
df = load_results(qs, ARMS, MODEL, DTYPE)
if SMOKE:
    analysis.peek(df)

## Accuracy

In [ ]:
analysis.accuracy(df, by=("dataset", "placement"))

## K1(m) − K1(0) by placement, equivalence under AB, and the interaction

In [ ]:
for m in STEPS[1:]:
    a = f"k1_m{m}"
    print(f"{a} − k1_m0")
    display(analysis.paired(df, a, "k1_m0", by=("dataset", "placement")))
    print("TOST under AB (split):")
    display(analysis.equivalent(df, a, "k1_m0", crossings=1))
    for ds in ("toy_d2", "musique"):
        print(f"{ds}: gap(AB) − gap(BB)")
        display(analysis.crossing_trend(df, a, "k1_m0", ds))

In [ ]:
gain = pd.concat([analysis.paired(df, f"k1_m{m}", "k1_m0", by=("dataset", "placement")).assign(m=m)
                  for m in STEPS[1:]]).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), sharey=True)
for ax, (ds, g) in zip(axes, gain.groupby("dataset")):
    for pl, gp in g.groupby("placement"):
        ax.errorbar(gp.m, gp["diff"], yerr=[gp["diff"] - gp.lo, gp.hi - gp["diff"]], marker="o", capsize=3, label=pl)
    ax.axhline(0, color="gray", lw=0.8)
    ax.set(title=ds, xlabel="latent steps m", xticks=STEPS[1:])
axes[0].set_ylabel("K1(m) − K1(0) (EM points, 95% CI)")
axes[0].legend()
plt.show()

## Positive control: T1 vs B1 (under BB, does text pre-composition work?)

In [ ]:
analysis.paired(df, "t1_1024", "b1", by=("dataset", "placement"))

## Findings

_(fill in after the full run)_